# AI_lab_2 — Kaggle / Colab
Увімкніть GPU та Internet і під'єднайте competition dataset. Перша клітинка завантажить код із https://github.com/yelgor/AI_lab_2; архів не потрібен.

Для двох незалежних GPU-середовищ виберіть різні моделі або folds, використовуючи ті самі manifests. На одному невеликому GPU запускайте одне навчання.


In [ ]:
from pathlib import Path
import os, sys, subprocess

print(f'Python {sys.version.split()[0]}', flush=True)
if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError('Потрібен Python 3.11, 3.12 або 3.13; перевірте версію середовища виконання.')

def bootstrap_run(arguments):
    # Forward pip/git output into the notebook instead of showing only
    # subprocess.CalledProcessError when an installation fails.
    with subprocess.Popen(arguments, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        returncode = process.wait()
    if returncode:
        raise RuntimeError(f'Команда завершилася з кодом {returncode}. Причина наведена вище у виводі pip/git.')

from IPython import get_ipython
_shell = get_ipython()
_is_colab = bool(os.environ.get('COLAB_RELEASE_TAG') or os.environ.get('COLAB_BACKEND_VERSION')) or (
    _shell is not None and type(_shell).__module__.startswith('google.colab'))
_is_kaggle = not _is_colab and (bool(os.environ.get('KAGGLE_KERNEL_RUN_TYPE')) or Path('/kaggle/working').is_dir())
if not _is_kaggle and not _is_colab:
    _is_colab = Path('/content').is_dir()
if not _is_kaggle and not _is_colab:
    raise RuntimeError('Цей notebook призначений для Kaggle або Colab.')
print('Середовище:', 'Kaggle' if _is_kaggle else 'Colab', flush=True)
PROJECT_DIR = Path('/kaggle/working/AI_lab_2' if _is_kaggle else '/content/AI_lab_2')
if not PROJECT_DIR.exists():
    bootstrap_run(['git', 'clone', 'https://github.com/yelgor/AI_lab_2.git', str(PROJECT_DIR)])
elif (PROJECT_DIR / '.git').is_dir():
    bootstrap_run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'])
assert (PROJECT_DIR / 'pyproject.toml').is_file(), f'У {PROJECT_DIR} немає pyproject.toml; перевірте каталог'
os.chdir(PROJECT_DIR)
extras = '.[kaggle]'
bootstrap_run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '-e', extras])


Після першого встановлення залежностей перезапустіть Python-сеанс Colab (Runtime → Restart session / Среда выполнения → Перезапустить сеанс), щоб kernel підхопив установлені версії. Потім продовжуйте з наступної клітинки `doctor`. Internet потрібен для встановлення пакетів і першого завантаження pretrained weights. Підтримуються Python 3.11–3.13.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, importlib
from cvlab import kaggle_data
kaggle_data = importlib.reload(kaggle_data)
RUNTIME = kaggle_data.detect_runtime()
PROJECT_DIR = Path('/kaggle/working/AI_lab_2' if RUNTIME == 'kaggle' else '/content/AI_lab_2')
os.chdir(PROJECT_DIR)
def run(*args):
    subprocess.run([sys.executable, '-m', 'cvlab', *map(str, args)], check=True)
run('doctor')


### Дані змагання: Kaggle / Colab

Змагання: https://www.kaggle.com/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026.

Клітинка спочатку шукає готові дані в `CVLAB_DATA_ROOT` і `/kaggle/input`, потім на локальному диску runtime. Якщо їх немає, завантажує competition через API.

- **Kaggle:** можна під'єднати competition через Add Input. Або додати Secret `KAGGLE_API_TOKEN` через Add-ons → Secrets і надати notebook доступ: дані завантажаться до `/kaggle/working/drone-data`.
- **Colab:** Secret `KAGGLE_API_TOKEN` з доступом notebook або приховане поле введення; каталог завантаження — `/content/drone-data`.

Перед завантаженням прийміть правила змагання у своєму Kaggle-акаунті. Токен отримайте на https://www.kaggle.com/settings/api → Generate New Token. Ключ не друкується й не записується в репозиторій.

Дані мають обсяг приблизно 11 GB. ZIP видаляється після успішного розпакування. Готові дані повторно не завантажуються. При HTTP 401 перевірте token, при HTTP 403 — участь і прийняття правил тим самим акаунтом.

Для діагностики збою завантаження запустіть `kaggle_data.check_access()` після першої клітинки встановлення: буде завантажено тільки малий test_images.csv. Оригінальна помилка CLI друкується з прихованим token.


In [ ]:
import os, importlib
from cvlab import kaggle_data
# Pick up a newer helper if git pull updated this clone in the same Python session.
kaggle_data = importlib.reload(kaggle_data)
DATA_ROOT = kaggle_data.ensure_data_root()
os.environ['CVLAB_DATA_ROOT'] = str(DATA_ROOT)
print('DATA_ROOT:', DATA_ROOT)


In [ ]:
import os
from cvlab.kaggle_data import ensure_data_root
DATA_ROOT = ensure_data_root()
print('DATA_ROOT:', DATA_ROOT)
os.environ['CVLAB_DATA_ROOT'] = str(DATA_ROOT)
run('prepare', '--root', DATA_ROOT, '--out', 'prepared', '--folds', '3', '--workers', '8')


Перегляньте розподіли та GT перед навчанням. Під час паралельної роботи використовуйте однаковий каталог `prepared/` в обох середовищах.

In [ ]:
from IPython.display import display, Image
print(json.dumps(json.loads(Path('prepared/eda_train/summary.json').read_text()), indent=2)[:5000])
display(Image(filename='prepared/eda_train/distributions.png'))
for path in sorted(Path('prepared/eda_train').glob('gt_*.jpg'))[:2]:
    display(Image(filename=str(path)))


In [ ]:
import yaml
MODEL = 'fcos'  # fasterrcnn / retinanet / fcos / rtdetr; variants див. configs/
FOLD = 0       # 0, 1, 2; кожний запускається незалежно
SMOKE = True   # Спочатку коротка перевірка. False — повний run.
config = yaml.safe_load(Path(f'configs/{MODEL}.yaml').read_text())
config['data'].update(root=str(DATA_ROOT), prepared='prepared', fold=FOLD)
name = f"{MODEL}_fold{FOLD}" + ('_smoke' if SMOKE else '')
config['output'] = f'runs/{name}'
if SMOKE:
    config['train'].update(epochs=1, max_batches_per_epoch=2)
# Якщо OOM: batch_size=1. Для порівняння моделей узгодьте параметри.
# config['loader']['batch_size'] = 1
config_path = Path('run_configs') / f'{name}.yaml'
config_path.parent.mkdir(exist_ok=True)
config_path.write_text(yaml.safe_dump(config, sort_keys=False))
print(config_path, config['output'])


Наступна клітинка запускає GPU-навчання. Smoke обмежує лише training batches; validation залишається повною. Повний run за замовчуванням має м’який ліміт 2 години і максимум 20 епох. Для повторного запуску задайте нове ім’я output або відновлюйте весь попередній run із `last.pt`.

In [ ]:
RESUME = False
arguments = ['train', '--config', config_path]
if RESUME:
    arguments += ['--resume', str(Path(config['output']) / 'last.pt')]
run(*arguments)


In [ ]:
import matplotlib.pyplot as plt
history = [json.loads(line) for line in (Path(config['output']) / 'history.jsonl').read_text().splitlines()]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot([r['elapsed_seconds']/3600 for r in history], [r['metrics']['coco100']['mAP'] for r in history], marker='o')
axes[0].set(xlabel='Elapsed hours', ylabel='COCO100 mAP', title=name)
axes[1].plot([r['epoch'] for r in history], [r['train_loss'] for r in history], marker='o')
axes[1].set(xlabel='Epoch', ylabel='Training loss')
plt.show()
print(json.dumps(json.loads((Path(config['output']) / 'best_metrics.json').read_text()), indent=2))


Для benchmark порівнюйте моделі на одному GPU, з однаковою precision, роздільністю та batch size. Batch latency не дорівнює затримці окремого кадру в онлайн-черзі.

In [ ]:
run('benchmark', '--config', config_path,
    '--checkpoint', Path(config['output']) / 'best.pt',
    '--set', 'loader.batch_size=1',
    '--out', Path(config['output']) / 'benchmark_batch1.json')


Submission генеруйте після повного навчання та вибору моделі. Ця клітинка не відправляє файл у змагання. Не використовуйте smoke checkpoint як фінальне рішення.

In [ ]:
GENERATE_SUBMISSION = False
if GENERATE_SUBMISSION:
    assert not SMOKE, 'Спершу виконайте повне навчання'
    run('predict', '--config', config_path,
        '--checkpoint', Path(config['output']) / 'best.pt',
        '--out', Path(config['output']) / 'submission.csv')


Збережіть весь каталог `runs/<name>/`, `prepared/` і `run_configs/` до завершення сесії. Для продовження відновіть їх за тими самими відносними шляхами. Реальні результати, failure cases і власні висновки ще потрібно отримати та проаналізувати; notebook не замінює авторський звіт.